In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

In [ ]:
df = pd.read_json('./other-analysis/data/marta-study-05-05-2026.json')

# cols are participant ids, rows are questions
df = df.transpose()

df.head()

In [ ]:
# get rid of lists in binary columns
binary_columns = ['academicReading', 'graphicDesign']
for col in binary_columns:
    df[col] = df[col].apply(lambda x: 'yes' if isinstance(x, list) and 'yes' in x else 'no')

In [ ]:
df.columns

In [ ]:
def getViewportMetrics(row):
    viewport_row = pd.DataFrame(columns=[
        "devicePixelRatio", "availScreenHeight", 
        "availScreenWidth", "screenHeight", 
        "screenWidth", "visualViewportHeight",
        "visualViewportWidth", "visualViewportScale",
        "windowInnerHeight", "windowInnerWidth",
        "windowOuterHeight", "windowOuterWidth"])

    if 'viewportMetrics' in row:
        viewport_data = row['viewportMetrics']
        devicePixelRatio = viewport_data.get('devicePixelRatio', np.nan)
        screenSettings = viewport_data.get('screen', {})
        availScreenHeight = screenSettings.get('availHeight', np.nan)
        availScreenWidth = screenSettings.get('availWidth', np.nan)
        screenHeight = screenSettings.get('height', np.nan)
        screenWidth = screenSettings.get('width', np.nan)
        visualViewport = viewport_data.get('visualViewport', {})
        visualViewportHeight = visualViewport.get('height', np.nan)
        visualViewportWidth = visualViewport.get('width', np.nan)
        visualViewportScale = visualViewport.get('scale', np.nan)
        windowSettings = viewport_data.get('window', {})
        windowInnerHeight = windowSettings.get('innerHeight', np.nan)
        windowInnerWidth = windowSettings.get('innerWidth', np.nan)
        windowOuterHeight = windowSettings.get('outerHeight', np.nan)
        windowOuterWidth = windowSettings.get('outerWidth', np.nan)

        viewport_row = pd.DataFrame({
            "devicePixelRatio": [devicePixelRatio],
            "availScreenHeight": [availScreenHeight],
            "availScreenWidth": [availScreenWidth],
            "screenHeight": [screenHeight],
            "screenWidth": [screenWidth],
            "visualViewportHeight": [visualViewportHeight],
            "visualViewportWidth": [visualViewportWidth],
            "visualViewportScale": [visualViewportScale],
            "windowInnerHeight": [windowInnerHeight],
            "windowInnerWidth": [windowInnerWidth],
            "windowOuterHeight": [windowOuterHeight],
            "windowOuterWidth": [windowOuterWidth]
        })
        return viewport_row
    return None

In [ ]:
invalid_counter = 0
for i, row in df.iterrows():
    try:
        viewport_metrics = getViewportMetrics(row)
        if viewport_metrics is not None:
            for col in viewport_metrics.columns:
                df.loc[i, col] = viewport_metrics.loc[0, col]

    except Exception as e:
        invalid_counter += 1

print(f"Number of invalid rows: {invalid_counter}")
df.head()

In [ ]:
# show rows where either sliderSettings and selectedLanguages is NaN
df[df['sliderSettings'].isna() & df['selectedLanguages'].isna()]['nativeLanguage'].value_counts()

In [ ]:
sns.scatterplot(x=df['visualViewportScale'], y=df['devicePixelRatio'], hue=df['devicePixelRatio'])

In [ ]:
df['testType'].value_counts()

In [ ]:
# show users where uiLanguage != nativeLanguage
df[df['uiLanguage'] != df['nativeLanguage']][['nativeLanguage']].value_counts()

In [ ]:
# show who picked Zulu as their native language
df[df['nativeLanguage'] == 'Zulu'][['nativeLanguage', 'uiLanguage']]

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(x='uiLanguage', data=df, order=df['uiLanguage'].value_counts().index)
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(16, 8))
sns.countplot(x='nativeLanguage', data=df, order=df['nativeLanguage'].value_counts().index)
plt.xticks(rotation=45)
plt.show()


In [ ]:
# from otherLanguages make a new column otherLanguageCount with the number of other languages
df['otherLanguageCount'] = df['otherLanguages'].apply(lambda x: len(x) if isinstance(x, list) else 0)

In [ ]:
sns.countplot(x='otherLanguageCount', data=df)

In [ ]:
df['otherLanguages'].value_counts()

In [ ]:
df['booksTimeframe'].value_counts()

In [ ]:
# convert booksRead to numeric
df['booksRead'] = pd.to_numeric(df['booksRead'], errors='coerce')

df['booksReadPerYear'] = df.apply(lambda row: row['booksRead'] * 12 if row['booksTimeframe'] == 'month' else (row['booksRead'] * 52 if row['booksTimeframe'] == 'week' else row['booksRead']), axis=1)
df['booksReadPerYear'] = pd.to_numeric(df['booksReadPerYear'], errors='coerce')

df['booksReadPerYear'].describe()

In [ ]:
# make new booksReadPerYearCategorical column based on booksReadPerYear

order = {'0': 0, '<1': 1.00001, '1-5': 5.00001, '6-10': 10.00001, '11-20': 20.00001, '21-50': 50.00001, '51-100': 100.00001, '>100': 101.00001}

def categorize_books_read(books_read):
    for category, threshold in order.items():
        if books_read <= threshold:
            return category
    return '>100'

df['booksReadPerYearCategorical'] = df['booksReadPerYear'].apply(categorize_books_read)

In [ ]:
sns.countplot(x='booksReadPerYearCategorical', data=df, order=order.keys())

In [ ]:
df['everydayReading'].value_counts()

In [ ]:
df['everydayReadingOther'].value_counts()

In [ ]:
sns.countplot(x='graphicDesign', data=df)

In [ ]:
df_slider = df.copy(deep=True)
df_slider = df_slider[pd.notna(df_slider['sliderSettings'])]

df_select = df.copy(deep=True)
df_select = df_select[pd.notna(df_select['selectedLanguages'])]

df_no_answer = df[~pd.notna(df['sliderSettings']) & ~pd.notna(df['selectedLanguages'])]

print(f'Total rows: {df.shape[0]}')
print(f'Total rows (slider): {df_slider.shape[0]}')
print(f'Total rows (select): {df_select.shape[0]}')
print(f'Total rows no answers: {df_no_answer.shape[0]}')


In [ ]:
import json
language_map = json.loads(open('./other-analysis/pilot-languages.json').read())
language_map = {lang: group for group, group_languages in language_map.items() for lang in group_languages}

### Processing slider test

In [ ]:
default_slider = {
    'lineHeight': 1.2,
    'letterSpacing': 0.0,
    'wordSpacing': 0.0,
}

In [ ]:
df_slider.columns

In [ ]:
df_slider_results_rowlike = pd.DataFrame(columns=["responseId", "testType", "nativeLanguage", "otherLanguages", "shownLanguage", "round", "isSameGroup", "isNativeLanguage","lineHeight_difference", "letterSpacing_difference", "wordSpacing_difference"])

for i, row in df_slider.iterrows():
    results_row = pd.DataFrame(columns=["responseId", "testType", "nativeLanguage", "otherLanguages", "shownLanguage", "round", "isSameGroup", "isNativeLanguage", "lineHeight_difference", "letterSpacing_difference", "wordSpacing_difference"], data=[[row.name, row['testType'], row['nativeLanguage'], row['otherLanguages'], "", np.nan, False, False, np.nan, np.nan, np.nan]])
    native_lang = row['nativeLanguage']
    questions = row['sliderSettings']
    for question in questions:
        difference_map = {
            'lineHeight': 0,
            'letterSpacing': 0,
            'wordSpacing': 0,
        }
        for val in default_slider.keys():
            value = float(question[val])
            if default_slider[val] != value:
                difference_map[val] = value - default_slider[val]
        for k, v in difference_map.items():
            results_row[f'{k}_difference'] = v

        results_row['shownLanguage'] = question['language']
        results_row['round'] = question['round']
        results_row['isSameGroup'] = (language_map.get(question['language']) == language_map.get(native_lang))
        results_row['isNativeLanguage'] = (question['language'] == native_lang)
        df_slider_results_rowlike = pd.concat([df_slider_results_rowlike, results_row], ignore_index=True)


df_slider_results_rowlike.head()
        


In [ ]:
import builtins

def getSliderResults(row):
    slider_row = pd.DataFrame(columns=[
        [f'{col}_{round_number}' for round_number in range(1, 9) for col in ['language', 'lineHeight', 'letterSpacing', 'wordSpacing', 'reasonText', 'isNativeLanguage', 'isSameGroup', 'timeSpent']]
    ])
    if 'sliderSettings' in row:
        slider_data = row['sliderSettings']
        timeStampDict = {}
        if 'taskTimestamps' in row:
            task_timestamps = row['taskTimestamps']
            for task in task_timestamps:
                startTime = datetime.datetime.strptime(task["start"], "%Y-%m-%dT%H:%M:%S.%fZ") if "start" in task else np.nan
                endTime = datetime.datetime.strptime(task["end"], "%Y-%m-%dT%H:%M:%S.%fZ") if "end" in task else np.nan
                timeSpent = endTime - startTime if pd.notna(startTime) and pd.notna(endTime) else np.nan
                timeStampDict[task['round']] = timeSpent.total_seconds() if pd.notna(timeSpent) else np.nan
        for test in slider_data:
            round_number = test.get('round', np.nan)
            language = test.get('language', np.nan)
            lineHeight = test.get('lineHeight', default_slider['lineHeight'])
            letterSpacing = test.get('letterSpacing', default_slider['letterSpacing'])
            wordSpacing = test.get('wordSpacing', default_slider['wordSpacing'])
            reasonText = test.get('reasonText', np.nan)
            isNativeLanguage = language == row['nativeLanguage']
            isSameGroup = language_map.get(language, 'Other') == language_map.get(row['nativeLanguage'], 'Other')
            slider_row.loc[0, f'language_{round_number}'] = language
            slider_row.loc[0, f'lineHeight_{round_number}'] = builtins.round(float(lineHeight), 2) if pd.notna(lineHeight) else np.nan
            slider_row.loc[0, f'letterSpacing_{round_number}'] = builtins.round(float(letterSpacing), 2) if pd.notna(letterSpacing) else np.nan
            slider_row.loc[0, f'wordSpacing_{round_number}'] = builtins.round(float(wordSpacing), 2) if pd.notna(wordSpacing) else np.nan
            slider_row.loc[0, f'reasonText_{round_number}'] = reasonText
            slider_row.loc[0, f'isNativeLanguage_{round_number}'] = isNativeLanguage
            slider_row.loc[0, f'isSameGroup_{round_number}'] = isSameGroup
            slider_row.loc[0, f'timeSpent_{round_number}'] = timeStampDict.get(str(round_number), np.nan)
        return slider_row
    return None

In [ ]:
df_slider_results = df_slider.copy(deep=True)

for i, row in df_slider_results.iterrows():
    viewport_metrics = getViewportMetrics(row)
    if viewport_metrics is not None:
        for col in viewport_metrics.columns:
            df_slider_results.loc[i, col] = viewport_metrics.loc[0, col]

    slider_results = getSliderResults(row)
    if slider_results is not None:
        for col in slider_results.columns:
            df_slider_results.loc[i, col] = slider_results.loc[0, col]
        
        if df_slider_results.loc[i, 'language_1'] == row['nativeLanguage']:
            df_slider_results.loc[i, 'sliderTestType'] = '1A'
        else:
            df_slider_results.loc[i, 'sliderTestType'] = '1B'
    



df_slider_results.head(3)

In [ ]:
# Change df_results_slider_rowlike testType to 1A/1B bsaed on df_results_slider index
df_slider_results_rowlike['testType'] = df_slider_results_rowlike['responseId'].apply(lambda x: df_slider_results.loc[x, "sliderTestType"])
df_slider_results_rowlike.head()

In [ ]:
df_slider_results_rowlike["testType"].value_counts()

In [ ]:
# Show how many results with nativeLanguage == 'Italian'
df_slider_results[df_slider_results['nativeLanguage'] == 'Italian'][['nativeLanguage']].value_counts()

In [ ]:
# Show where isSameGroup_1 is True
df_slider_results[df_slider_results['isNativeLanguage_1'] == False][['nativeLanguage', 'language_1', 'isSameGroup_1']]

In [ ]:
df_slider_results.columns

### Word Spacing
##### Native language vs not native language

In [ ]:
sns.violinplot(x=df_slider_results_rowlike['isNativeLanguage'], y=df_slider_results_rowlike['wordSpacing_difference'], hue=df_slider_results_rowlike['isNativeLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

##### Same group vs not same group

In [ ]:
sns.violinplot(x=df_slider_results_rowlike['isSameGroup'], y=df_slider_results_rowlike['wordSpacing_difference'], hue=df_slider_results_rowlike['isSameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### Line height
Default = 1.2

In [ ]:
sns.violinplot(x=df_slider_results_rowlike['isNativeLanguage'], y=df_slider_results_rowlike['lineHeight_difference'], hue=df_slider_results_rowlike['isNativeLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.violinplot(x=df_slider_results_rowlike['isSameGroup'], y=df_slider_results_rowlike['lineHeight_difference'], hue=df_slider_results_rowlike['isSameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

#### Letter spacing

In [ ]:
sns.violinplot(x=df_slider_results_rowlike['isNativeLanguage'], y=df_slider_results_rowlike['letterSpacing_difference'], hue=df_slider_results_rowlike['isNativeLanguage'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sns.violinplot(x=df_slider_results_rowlike['isSameGroup'], y=df_slider_results_rowlike['letterSpacing_difference'], hue=df_slider_results_rowlike['isSameGroup'])

# legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
columns_to_drop = ['taskTimestamps', 'sliderSettings', 'booksRead', 'booksTimeframe', 'selectedLanguages', 'otherLanguageCount', 'booksReadPerYearCategorical', 'viewportMetrics', 'languageRounds', 'availScreenHeight', 'availScreenWidth', 'visualViewportHeight', 'visualViewportWidth', 'windowInnerHeight', 'windowInnerWidth', 'windowOuterHeight', 'windowOuterWidth']
df_slider_results = df_slider_results.drop(columns=columns_to_drop)

for i, row in df_slider_results.iterrows():
    totalTimeSpent = 0
    start = datetime.datetime.strptime(row['fullTaskStart'], "%Y-%m-%dT%H:%M:%S.%fZ") if 'fullTaskStart' in df_slider_results.columns and pd.notna(row['fullTaskStart']) else None
    end = datetime.datetime.strptime(row['fullTaskEnd'], "%Y-%m-%dT%H:%M:%S.%fZ") if 'fullTaskEnd' in df_slider_results.columns and pd.notna(row['fullTaskEnd']) else None

    if start and end:
        df_slider_results.at[i, 'totalTimeSpent'] = (end - start).total_seconds()
    else:
        df_slider_results.at[i, 'totalTimeSpent'] = np.nan

df_slider_results = df_slider_results.drop(columns=['fullTaskStart', 'fullTaskEnd'])

df_slider_results.head()

In [ ]:
df_slider_results_rowlike.to_csv('./other-analysis/data/slider_results_rowlike-05-05-2026.csv', index=False)
df_slider_results.to_csv('./other-analysis/data/slider_results-05-05-2026.csv', index=False)

### Select test

In [ ]:
df_select.columns

In [ ]:
df_select_results_rowlike = pd.DataFrame(columns=["responseId", "nativeLanguage", "otherLanguages", "round", "shownLanguage1", "shownLanguage2", "shownLanguage3", "shownLanguage4", "firstSelectedLanguage", "secondSelectedLanguage", "selectedMatch", "ifMatchSameGroupAsNative", "nativeInSelection"])

for i, row in df_select.iterrows():
    results_row = pd.DataFrame(columns=["responseId", "nativeLanguage", "otherLanguages", "round", "shownLanguage1", "shownLanguage2", "shownLanguage3", "shownLanguage4" , "firstSelectedLanguage", "secondSelectedLanguage", "selectedMatch", "ifMatchSameGroupAsNative", "nativeInSelection"], data=[[row.name, row['nativeLanguage'], row['otherLanguages'], np.nan, "", "", "", "", "", "", False, False, False]])
    native_lang = row['nativeLanguage']
    questions = row['selectedLanguages']
    for question in questions:
        # Somehow there are questions with less than 2 selections or even ZERO selections??
        if 'selections' not in question or len(question['selections']) < 2:
            continue
        try:
            first_lang = question['selections'][0]['language']
            second_lang = question['selections'][1]['language']
            results_row['firstSelectedLanguage'] = first_lang
            results_row['secondSelectedLanguage'] = second_lang

            results_row['shownLanguage1'] = question['all_languages'][0]
            results_row['shownLanguage2'] = question['all_languages'][1]
            results_row['shownLanguage3'] = question['all_languages'][2]
            results_row['shownLanguage4'] = question['all_languages'][3]

            results_row['round'] = question['round']
        except:
            print(row.name, question)

        # print(first_lang, second_lang, native_lang)

        
        # Edge case
        if first_lang == 'Haitian Creole':
            first_lang = 'Creole'

        if second_lang == 'Haitian Creole':
            second_lang = 'Creole'

        selected_match = (language_map[first_lang] == language_map[second_lang])
        results_row['selectedMatch'] = selected_match


        if selected_match:
            results_row['ifMatchSameGroupAsNative'] = (language_map.get(first_lang) == language_map.get(native_lang))

        if native_lang == first_lang or native_lang == second_lang:
            results_row['nativeInSelection'] = True

        df_select_results_rowlike = pd.concat([df_select_results_rowlike, results_row], ignore_index=True)


df_select_results_rowlike.head()


In [ ]:
sns.countplot(x=df_select_results_rowlike['selectedMatch'])

In [ ]:
sns.countplot(x=df_select_results_rowlike['selectedMatch'], hue=df_select_results_rowlike['nativeInSelection'])

In [ ]:
sns.countplot(x=df_select_results_rowlike['selectedMatch'], hue=df_select_results_rowlike['ifMatchSameGroupAsNative'])

In [ ]:
# a 4x4 grid of countplots showing firstSelectedLanguage == shownLanguage1-4
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.countplot(x=df_select_results_rowlike['firstSelectedLanguage'] == df_select_results_rowlike['shownLanguage1'],hue=df_select_results_rowlike['selectedMatch'], ax=axes[0, 0])
sns.countplot(x=df_select_results_rowlike['firstSelectedLanguage'] == df_select_results_rowlike['shownLanguage2'],hue=df_select_results_rowlike['selectedMatch'], ax=axes[0, 1])
sns.countplot(x=df_select_results_rowlike['firstSelectedLanguage'] == df_select_results_rowlike['shownLanguage3'],hue=df_select_results_rowlike['selectedMatch'], ax=axes[1, 0])
# Last plot can never be True
plt.show()

In [ ]:
# a 4x4 grid of countplots showing secondSelectedLanguage == shownLanguage1-4
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
# First plot can never be True
sns.countplot(x=df_select_results_rowlike['secondSelectedLanguage'] == df_select_results_rowlike['shownLanguage2'], hue=df_select_results_rowlike['selectedMatch'], ax=axes[0, 1])
sns.countplot(x=df_select_results_rowlike['secondSelectedLanguage'] == df_select_results_rowlike['shownLanguage3'], hue=df_select_results_rowlike['selectedMatch'], ax=axes[1, 0])
sns.countplot(x=df_select_results_rowlike['secondSelectedLanguage'] == df_select_results_rowlike['shownLanguage4'], hue=df_select_results_rowlike['selectedMatch'], ax=axes[1, 1])
plt.show()

In [ ]:
# save to csv
df_select_results_rowlike.to_csv('./other-analysis/data/select_results.csv', index=False)

In [ ]:
print("\nFirst selected lang")
print(f'1st: {df_select_results_rowlike[df_select_results_rowlike['firstSelectedLanguage'] == df_select_results_rowlike['shownLanguage1']].shape}')
print(f'2nd: {df_select_results_rowlike[df_select_results_rowlike['firstSelectedLanguage'] == df_select_results_rowlike['shownLanguage2']].shape}')
print(f'3rd: {df_select_results_rowlike[df_select_results_rowlike['firstSelectedLanguage'] == df_select_results_rowlike['shownLanguage3']].shape}')
print(f'4th: {df_select_results_rowlike[df_select_results_rowlike['firstSelectedLanguage'] == df_select_results_rowlike['shownLanguage4']].shape}')

print("\nSecond selected lang")
print(f'1st: {df_select_results_rowlike[df_select_results_rowlike['secondSelectedLanguage'] == df_select_results_rowlike['shownLanguage1']].shape}')
print(f'2nd: {df_select_results_rowlike[df_select_results_rowlike['secondSelectedLanguage'] == df_select_results_rowlike['shownLanguage2']].shape}')
print(f'3rd: {df_select_results_rowlike[df_select_results_rowlike['secondSelectedLanguage'] == df_select_results_rowlike['shownLanguage3']].shape}')
print(f'4th: {df_select_results_rowlike[df_select_results_rowlike['secondSelectedLanguage'] == df_select_results_rowlike['shownLanguage4']].shape}')

In [ ]:
df_select_results = df_select.copy(deep=True)

def getSelectResults(row):
    select_row = pd.DataFrame(columns=[
        f'{col}_{round_number}' for round_number in range(1, 9) for col in ['shownLanguages', 'firstSelectedLanguage', 'secondSelectedLanguage', 'selectedMatch', 'nativeGroup']
    ])

    if 'selectedLanguages' in row:
        select_data = row['selectedLanguages']

        timeStampDict = {}
        if 'taskTimestamps' in row:
            task_timestamps = row['taskTimestamps']
            for task in task_timestamps:
                startTime = datetime.datetime.strptime(task["start"], "%Y-%m-%dT%H:%M:%S.%fZ") if "start" in task else np.nan
                endTime = datetime.datetime.strptime(task["end"], "%Y-%m-%dT%H:%M:%S.%fZ") if "end" in task else np.nan
                timeSpent = endTime - startTime if pd.notna(startTime) and pd.notna(endTime) else np.nan
                timeStampDict[task['round']] = timeSpent.total_seconds() if pd.notna(timeSpent) else np.nan

        for question in select_data:
            round_number = question.get('round', np.nan)
            all_languages = question.get('all_languages', [np.nan]*4)
            selections = question.get('selections', [])
            firstSelectedLanguage = selections[0]['language'] if len(selections) > 0 else np.nan
            secondSelectedLanguage = selections[1]['language'] if len(selections) > 1 else np.nan
            selectedMatch = (language_map.get(firstSelectedLanguage) == language_map.get(secondSelectedLanguage)) if (firstSelectedLanguage and secondSelectedLanguage) and pd.notna(firstSelectedLanguage) and pd.notna(secondSelectedLanguage) else False
            ifMatchSameGroupAsNative = ((language_map.get(firstSelectedLanguage) == language_map.get(row['nativeLanguage'])) or (language_map.get(secondSelectedLanguage) == language_map.get(row['nativeLanguage']))) if selectedMatch and firstSelectedLanguage and pd.notna(firstSelectedLanguage) and pd.notna(secondSelectedLanguage) else False
        
            select_row.loc[0, f'shownLanguages_{round_number}'] = str(all_languages)
            select_row.loc[0, f'firstSelectedLanguage_{round_number}'] = firstSelectedLanguage
            select_row.loc[0, f'secondSelectedLanguage_{round_number}'] = secondSelectedLanguage
            select_row.loc[0, f'selectedMatch_{round_number}'] = selectedMatch
            select_row.loc[0, f'nativeGroup_{round_number}'] = ifMatchSameGroupAsNative
            select_row.loc[0, f'timeSpent_{round_number}'] = timeStampDict.get(str(round_number), np.nan)

        return select_row
    return None

for i, row in df_select_results.iterrows():
    select_results = getSelectResults(row)
    if select_results is not None:
        for col in select_results.columns:
            df_select_results.loc[i, col] = select_results.loc[0, col]


df_select_results.head()


In [ ]:
df_select_results.columns

In [ ]:
columns_to_drop = ['taskTimestamps', 'sliderSettings', 'booksRead', 'booksTimeframe', 'selectedLanguages', 'otherLanguageCount', 'booksReadPerYearCategorical', 'viewportMetrics', 'languageRounds', 'availScreenHeight', 'availScreenWidth', 'visualViewportHeight', 'visualViewportWidth', 'windowInnerHeight', 'windowInnerWidth', 'windowOuterHeight', 'windowOuterWidth']
df_select_results = df_select_results.drop(columns=columns_to_drop)

for i, row in df_select_results.iterrows():
    totalTimeSpent = 0
    start = datetime.datetime.strptime(row['fullTaskStart'], "%Y-%m-%dT%H:%M:%S.%fZ") if 'fullTaskStart' in df_select_results.columns and pd.notna(row['fullTaskStart']) else None
    end = datetime.datetime.strptime(row['fullTaskEnd'], "%Y-%m-%dT%H:%M:%S.%fZ") if 'fullTaskEnd' in df_select_results.columns and pd.notna(row['fullTaskEnd']) else None

    if start and end:
        df_select_results.at[i, 'totalTimeSpent'] = (end - start).total_seconds()
    else:
        df_select_results.at[i, 'totalTimeSpent'] = np.nan

df_select_results = df_select_results.drop(columns=['fullTaskStart', 'fullTaskEnd'])

df_select_results.head()

In [ ]:
df_select_results.shape

In [ ]:
df_select_results.to_csv('./other-analysis/data/select_results-05-05-2026.csv', index=False)

In [ ]:
# Round 1: show only the most frequent shown-language combinations (readable view)
top_n = 25
counts = df_select_results['shownLanguages_1'].value_counts(dropna=False)
top_counts = counts.head(top_n).sort_values(ascending=True)

plt.figure(figsize=(14, 10))
ax = sns.barplot(x=top_counts.values, y=top_counts.index, orient='h', color='#1f77b4')
ax.set_title(f'Round 1 shownLanguages_1 — Top {top_n} combinations')
ax.set_xlabel('Count')
ax.set_ylabel('shownLanguages_1 combination')

total = counts.sum()
for i, value in enumerate(top_counts.values):
    pct = (value / total) * 100 if total else 0
    ax.text(value + 0.2, i, f'{value} ({pct:.1f}%)', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Optional: language-level frequency in round 1 (more interpretable than combination-level)
import ast

round1_languages = (
    df_select_results['shownLanguages_1']
    .dropna()
    .apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
    .replace('', np.nan)
    .dropna()
 )

language_counts = round1_languages.value_counts().sort_values(ascending=True)

plt.figure(figsize=(12, 8))
ax = sns.barplot(x=language_counts.values, y=language_counts.index, orient='h', color='#2a9d8f')
ax.set_title('Round 1 individual language frequency')
ax.set_xlabel('Count')
ax.set_ylabel('Language')

for i, value in enumerate(language_counts.values):
    ax.text(value + 0.2, i, str(value), va='center')

plt.tight_layout()
plt.show()

In [ ]:
# TEMP dataframe: combine shownLanguages_(1-10) across all available rounds
import re
import ast

def canonicalize_shown_languages(value):
    if isinstance(value, list):
        parsed = value
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return np.nan
        try:
            parsed = ast.literal_eval(value)
        except Exception:
            parsed = [value]
    else:
        return np.nan

    if not isinstance(parsed, list):
        parsed = [parsed]

    cleaned = [str(x).strip() for x in parsed if pd.notna(x) and str(x).strip()]
    if not cleaned:
        return np.nan

    # Sort so equivalent combinations are counted together, independent of display order
    return str(sorted(cleaned))

round_cols = []
for col in df_select_results.columns:
    match = re.match(r'^shownLanguages?_(\d+)$', col)
    if match and 1 <= int(match.group(1)) <= 10:
        round_cols.append(col)

round_cols = sorted(round_cols, key=lambda c: int(c.split('_')[-1]))
print(f'Using round columns: {round_cols}')

temp_shown_combinations = df_select_results[round_cols].copy()

long_df = (
    temp_shown_combinations     
    .melt(var_name='round_col', value_name='shownLanguages')
    .dropna(subset=['shownLanguages'])
 )
long_df['shownLanguages'] = long_df['shownLanguages'].apply(canonicalize_shown_languages)
long_df = long_df.dropna(subset=['shownLanguages'])
long_df['round'] = long_df['round_col'].str.extract(r'_(\d+)$').astype(int)

top_n = 48 # only 48 options anyways
counts = long_df['shownLanguages'].value_counts()
top_counts = counts.head(top_n).sort_values(ascending=True)

plt.figure(figsize=(14, 10))
ax = sns.barplot(x=top_counts.values, y=top_counts.index, orient='h', color='#1f77b4')
ax.set_title(f'All rounds (1-10): Top {top_n} shownLanguages combinations')
ax.set_xlabel('Count')
ax.set_ylabel('shownLanguages combination')

total = counts.sum()
for i, value in enumerate(top_counts.values):
    pct = (value / total) * 100 if total else 0
    ax.text(value + 0.2, i, f'{value} ({pct:.1f}%)', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Same analysis at language level across all rounds (1-10)
def parse_language_combo(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else [str(parsed)]
        except Exception:
            return [value]
    return []

all_shown_languages = (
    long_df['shownLanguages']
    .apply(parse_language_combo)
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
 )
all_shown_languages = all_shown_languages[all_shown_languages != '']

language_counts = all_shown_languages.value_counts().sort_values(ascending=True)

plt.figure(figsize=(12, 8))
ax = sns.barplot(x=language_counts.values, y=language_counts.index, orient='h', color='#2a9d8f')
ax.set_title('All rounds (1-10): Individual language frequency')
ax.set_xlabel('Count')
ax.set_ylabel('Language')

for i, value in enumerate(language_counts.values):
    ax.text(value + 0.2, i, str(value), va='center')

plt.tight_layout()
plt.show()

# temp dataframes available for quick checks
print(f'temp_shown_combinations shape: {temp_shown_combinations.shape}')
print(f'long_df shape: {long_df.shape}')

##### Same as above but only for Dutch results

In [ ]:
# TEMP dataframe: combine shownLanguages_(1-10) across all available rounds
import re
import ast

def canonicalize_shown_languages(value):
    if isinstance(value, list):
        parsed = value
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return np.nan
        try:
            parsed = ast.literal_eval(value)
        except Exception:
            parsed = [value]
    else:
        return np.nan

    if not isinstance(parsed, list):
        parsed = [parsed]

    cleaned = [str(x).strip() for x in parsed if pd.notna(x) and str(x).strip()]
    if not cleaned:
        return np.nan

    # Sort so equivalent combinations are counted together, independent of display order
    return str(sorted(cleaned))

round_cols = []
for col in df_select_results.columns:
    match = re.match(r'^shownLanguages?_(\d+)$', col)
    if match and 1 <= int(match.group(1)) <= 10:
        round_cols.append(col)

round_cols = sorted(round_cols, key=lambda c: int(c.split('_')[-1]))
print(f'Using round columns: {round_cols}')

temp_shown_combinations = df_select_results[df_select_results['nativeLanguage'] == 'Dutch'][round_cols].copy()

long_df = (
    temp_shown_combinations     
    .melt(var_name='round_col', value_name='shownLanguages')
    .dropna(subset=['shownLanguages'])
 )
long_df['shownLanguages'] = long_df['shownLanguages'].apply(canonicalize_shown_languages)
long_df = long_df.dropna(subset=['shownLanguages'])
long_df['round'] = long_df['round_col'].str.extract(r'_(\d+)$').astype(int)

top_n = 48 # only 48 options anyways
counts = long_df['shownLanguages'].value_counts()
top_counts = counts.head(top_n).sort_values(ascending=True)

plt.figure(figsize=(14, 10))
ax = sns.barplot(x=top_counts.values, y=top_counts.index, orient='h', color='#1f77b4')
ax.set_title(f'All rounds (1-10): Top {top_n} shownLanguages combinations')
ax.set_xlabel('Count')
ax.set_ylabel('shownLanguages combination')

total = counts.sum()
for i, value in enumerate(top_counts.values):
    pct = (value / total) * 100 if total else 0
    ax.text(value + 0.2, i, f'{value} ({pct:.1f}%)', va='center')

plt.tight_layout()
plt.show()